In [93]:
import os
import random
from pathlib import Path

import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from skimage.feature import graycomatrix, graycoprops

# ── Konfigurasi ──────────────────────────────────────────────
DATASET_DIR = Path("Dataset")
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

CLASS_NAMES = sorted([
    d for d in os.listdir(DATASET_DIR)
    if (DATASET_DIR / d).is_dir()
])

print("Kelas ditemukan:", CLASS_NAMES)
print("Jumlah kelas   :", len(CLASS_NAMES))

Kelas ditemukan: ['100k', '10k', '20k', '2k', '50k', '5k']
Jumlah kelas   : 6


In [94]:
TARGET_WIDTH = 256
TARGET_HEIGHT = 256

USE_HSV = False
USE_RGB = True
USE_GLCM = True

if USE_HSV and USE_RGB:
    raise ValueError(
        "HSV dan RGB tidak boleh digunakan bersamaan."
    )

if not (USE_HSV or USE_RGB):
    raise ValueError(
        "Pilih minimal HSV atau RGB."
    )

if not (USE_HSV or USE_RGB or USE_GLCM):
    raise ValueError(
        "Pilih minimal satu jenis fitur."
    )

feature_parts = []

if USE_HSV:
    feature_parts.append("hsv")

if USE_RGB:
    feature_parts.append("rgb")

if USE_GLCM:
    feature_parts.append("glcm")

FEATURES_NAME = "_".join(feature_parts)

print("="*40)
print("Feature Configuration")
print("="*40)
print("HSV  :", USE_HSV)
print("RGB  :", USE_RGB)
print("GLCM :", USE_GLCM)
print("NAME :", FEATURES_NAME)
print("="*40)

Feature Configuration
HSV  : False
RGB  : True
GLCM : True
NAME : rgb_glcm


# Preprocessing

In [95]:
def mask_hsv(image_bgr, gray_threshold=25, min_area_ratio=0.01, kernel_size=7):
    if image_bgr is None:
        raise ValueError("Gambar tidak berhasil dibaca.")

    work_img = image_bgr.copy()

    # Konversi ke grayscale
    gray = cv2.cvtColor(work_img, cv2.COLOR_BGR2GRAY)

    # Threshold
    _, mask = cv2.threshold(
        gray,
        gray_threshold,
        255,
        cv2.THRESH_BINARY
    )

    # Morphological operations
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (kernel_size, kernel_size)
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel,
        iterations=1
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=2
    )

    # Fill holes
    h, w = mask.shape

    flood = mask.copy()
    flood_mask = np.zeros((h + 2, w + 2), np.uint8)

    cv2.floodFill(
        flood,
        flood_mask,
        (0, 0),
        255
    )

    flood_inv = cv2.bitwise_not(flood)
    mask = mask | flood_inv

    # Ambil komponen terbesar
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask,
        connectivity=8
    )

    refined = np.zeros_like(mask)

    if num_labels > 1:
        areas = stats[1:, cv2.CC_STAT_AREA]
        largest_idx = 1 + np.argmax(areas)
        largest_area = stats[largest_idx, cv2.CC_STAT_AREA]

        min_area = int(min_area_ratio * h * w)

        if largest_area >= min_area:
            refined[labels == largest_idx] = 255
        else:
            refined = mask.copy()
    else:
        refined = mask.copy()

    # Segmentasi
    result = cv2.bitwise_and(
        work_img,
        work_img,
        mask=refined
    )

    return work_img, refined, result

In [96]:
def crop_to_mask(image_bgr, mask, pad_ratio=0.05):

    coords = cv2.findNonZero(mask)

    if coords is None:
        return image_bgr

    x, y, w, h = cv2.boundingRect(coords)

    h_img, w_img = image_bgr.shape[:2]

    pad = int(max(w, h) * pad_ratio)

    x1 = max(x - pad, 0)
    y1 = max(y - pad, 0)
    x2 = min(x + w + pad, w_img)
    y2 = min(y + h + pad, h_img)

    return image_bgr[y1:y2, x1:x2]

In [97]:
def resize(image_bgr, target_width, target_height, fill_value=0):
    h, w = image_bgr.shape[:2]

    # Hitung skala
    scale = min(target_width / w, target_height / h)

    new_w = int(round(w * scale))
    new_h = int(round(h * scale))

    # Resize
    resized = cv2.resize(
        image_bgr,
        (new_w, new_h),
        interpolation=cv2.INTER_AREA
    )

    # Buat canvas
    output = np.full(
        (target_height, target_width, 3),
        fill_value,
        dtype=np.uint8
    )

    # Posisi tengah
    x_offset = (target_width - new_w) // 2
    y_offset = (target_height - new_h) // 2

    # Tempel gambar
    output[
        y_offset:y_offset+new_h,
        x_offset:x_offset+new_w
    ] = resized

    return output

In [98]:
def normalize_minmax(image_bgr):
    return image_bgr.astype(np.float32) / 255.0

# Ekstraksi Fitur Warna dengan Histogram HSV

In [99]:
def extract_hsv_hist_features(
    image_bgr,
    grid_size=(2, 2),   # 4-kuadran
    bins_h=8,
    bins_s=8,
    bins_v=8
):
    
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    h, w = hsv.shape[:2]
    gh, gw = grid_size

    features = []

    for i in range(gh):
        for j in range(gw):
            y0 = int(i * h / gh)
            y1 = int((i + 1) * h / gh)
            x0 = int(j * w / gw)
            x1 = int((j + 1) * w / gw)

            cell = hsv[y0:y1, x0:x1]

            hist_h = cv2.calcHist([cell], [0], None, [bins_h], [0, 180])
            hist_s = cv2.calcHist([cell], [1], None, [bins_s], [0, 256])
            hist_v = cv2.calcHist([cell], [2], None, [bins_v], [0, 256])

            hist_h = cv2.normalize(hist_h, None).flatten()
            hist_s = cv2.normalize(hist_s, None).flatten()
            hist_v = cv2.normalize(hist_v, None).flatten()

            features.extend(hist_h)
            features.extend(hist_s)
            features.extend(hist_v)

    return np.array(features, dtype=np.float32)

# Ekstraksi Fitur Warna dengan RGB

In [100]:
def extract_rgb_hist_features(
    image_bgr,
    grid_size=(2, 2),
    bins_r=8,
    bins_g=8,
    bins_b=8
):

    h, w = image_bgr.shape[:2]
    gh, gw = grid_size

    features = []

    for i in range(gh):
        for j in range(gw):

            y0 = int(i * h / gh)
            y1 = int((i + 1) * h / gh)

            x0 = int(j * w / gw)
            x1 = int((j + 1) * w / gw)

            cell = image_bgr[y0:y1, x0:x1]

            # OpenCV: B G R
            hist_b = cv2.calcHist(
                [cell],
                [0],
                None,
                [bins_b],
                [0, 256]
            )

            hist_g = cv2.calcHist(
                [cell],
                [1],
                None,
                [bins_g],
                [0, 256]
            )

            hist_r = cv2.calcHist(
                [cell],
                [2],
                None,
                [bins_r],
                [0, 256]
            )

            hist_b = cv2.normalize(
                hist_b,
                None
            ).flatten()

            hist_g = cv2.normalize(
                hist_g,
                None
            ).flatten()

            hist_r = cv2.normalize(
                hist_r,
                None
            ).flatten()

            features.extend(hist_b)
            features.extend(hist_g)
            features.extend(hist_r)

    return np.array(
        features,
        dtype=np.float32
    )

# Ekstraksi Fitur Tekstur dengan GLCM

In [101]:
def extract_glcm_features(
    image_bgr,
    distances=(1, 2, 3),
    angles=(0, np.pi / 4, np.pi / 2, 3 * np.pi / 4),
    levels=32
):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

    # Kuantisasi grayscale ke levels tertentu
    gray_q = np.floor(gray.astype(np.float32) * levels / 256.0).astype(np.uint8)
    gray_q = np.clip(gray_q, 0, levels - 1)

    glcm = graycomatrix(
        gray_q,
        distances=list(distances),
        angles=list(angles),
        levels=levels,
        symmetric=True,
        normed=True
    )

    props = ["contrast", "homogeneity", "energy", "correlation"]
    features = []

    for prop in props:
        values = graycoprops(glcm, prop)
        features.extend(values.flatten())

    return np.array(features, dtype=np.float32)

# Prepro + Ekstrak

In [102]:
def preprocess_for_features(image_bgr):

    original, mask, segmented = mask_hsv(image_bgr)

    cropped = crop_to_mask(
        segmented,
        mask
    )

    resized = resize(
        cropped,
        TARGET_WIDTH,
        TARGET_HEIGHT
    )

    return resized

In [103]:
def extract_features(
    image_bgr,
    use_hsv=USE_HSV,
    use_rgb=USE_RGB,
    use_glcm=USE_GLCM
):


    features = []

    if use_hsv:
        features.extend(
            extract_hsv_hist_features(image_bgr)
        )

    if use_rgb:
        features.extend(
            extract_rgb_hist_features(image_bgr)
        )

    if use_glcm:
        features.extend(
            extract_glcm_features(image_bgr)
        )

    return np.array(
        features,
        dtype=np.float32
    )

# Buat Nama Fitur

In [104]:
def generate_feature_names(
    use_hsv=USE_HSV,
    use_rgb=USE_RGB,
    use_glcm=USE_GLCM,
    grid_size=(2,2),
    bins=8,
    distances=(1,2,3),
    angles=(0,np.pi/4,np.pi/2,3*np.pi/4)
):

    feature_names=[]

    gh,gw=grid_size

    if use_hsv:

        quadrant=1

        for i in range(gh):
            for j in range(gw):

                for b in range(bins):
                    feature_names.append(
                        f"H_Q{quadrant}_B{b+1}"
                    )

                for b in range(bins):
                    feature_names.append(
                        f"S_Q{quadrant}_B{b+1}"
                    )

                for b in range(bins):
                    feature_names.append(
                        f"V_Q{quadrant}_B{b+1}"
                    )

                quadrant+=1

    if use_rgb:

        quadrant=1

        for i in range(gh):
            for j in range(gw):

                for b in range(bins):
                    feature_names.append(
                        f"B_Q{quadrant}_B{b+1}"
                    )

                for b in range(bins):
                    feature_names.append(
                        f"G_Q{quadrant}_B{b+1}"
                    )

                for b in range(bins):
                    feature_names.append(
                        f"R_Q{quadrant}_B{b+1}"
                    )

                quadrant+=1

    if use_glcm:

        props=[
            "contrast",
            "homogeneity",
            "energy",
            "correlation"
        ]

        angle_names={
            0:"0",
            np.pi/4:"45",
            np.pi/2:"90",
            3*np.pi/4:"135"
        }

        for prop in props:
            for d in distances:
                for a in angles:

                    feature_names.append(
                        f"{prop}_d{d}_a{angle_names[a]}"
                    )

    return feature_names

In [105]:
feature_names = generate_feature_names()

print("Jumlah nama fitur:", len(feature_names))

print("\n10 nama fitur pertama:")
print(feature_names[:10])

Jumlah nama fitur: 144

10 nama fitur pertama:
['B_Q1_B1', 'B_Q1_B2', 'B_Q1_B3', 'B_Q1_B4', 'B_Q1_B5', 'B_Q1_B6', 'B_Q1_B7', 'B_Q1_B8', 'G_Q1_B1', 'G_Q1_B2']


# Jadikan Dataframe

In [106]:
feature_names = generate_feature_names(
    use_hsv=USE_HSV,
    use_rgb=USE_RGB,
    use_glcm=USE_GLCM
)

dataset_rows=[]

for class_name in CLASS_NAMES:

    class_dir=DATASET_DIR/class_name

    image_files=sorted([
        f for f in class_dir.iterdir()
        if f.suffix.lower() in (
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp",
            ".webp"
        )
    ])

    for image_file in tqdm(
        image_files,
        desc=class_name
    ):

        image=cv2.imread(
            str(image_file)
        )

        if image is None:
            continue

        processed=preprocess_for_features(
            image
        )

        extracted_features=extract_features(
            processed,
            use_hsv=USE_HSV,
            use_rgb=USE_RGB,
            use_glcm=USE_GLCM
        )

        assert len(feature_names)==len(extracted_features)

        row={
            "filename":image_file.name,
            "label":class_name
        }

        row.update(
            dict(
                zip(
                    feature_names,
                    extracted_features
                )
            )
        )

        dataset_rows.append(
            row
        )

feature_df=pd.DataFrame(
    dataset_rows
)

print(feature_df.shape)

feature_df.head()

5k: 100%|██████████| 32/32 [00:05<00:00,  5.85it/s]

(192, 146)


,filename,label,B_Q1_B1,B_Q1_B2,B_Q1_B3,B_Q1_B4,B_Q1_B5,B_Q1_B6,B_Q1_B7,B_Q1_B8,...,correlation_d1_a90,correlation_d1_a135,correlation_d2_a0,correlation_d2_a45,correlation_d2_a90,correlation_d2_a135,correlation_d3_a0,correlation_d3_a45,correlation_d3_a90,correlation_d3_a135
0,100000_sample_ (1).jpg,100k,0.911242,0.017922,0.014839,0.014743,0.096645,0.155326,0.367985,0.0,...,0.991089,0.979421,0.967998,0.979754,0.980559,0.979421,0.950121,0.954202,0.971901,0.953769
1,100000_sample_ (10).jpg,100k,0.862024,0.010829,0.037429,0.105663,0.333602,0.364512,0.008516,0.0,...,0.990465,0.978796,0.968332,0.979177,0.981332,0.978796,0.951530,0.955657,0.973292,0.954916
2,100000_sample_ (11).jpg,100k,0.954914,0.004058,0.028219,0.116932,0.146349,0.225933,0.034489,0.0,...,0.991861,0.980943,0.971543,0.982211,0.981542,0.980943,0.955597,0.958445,0.972042,0.956086
3,100000_sample_ (12).jpg,100k,0.774196,0.004858,0.015992,0.024697,0.156378,0.608102,0.074090,0.0,...,0.991168,0.979711,0.968457,0.979057,0.982071,0.979711,0.951477,0.954519,0.973748,0.955960
4,100000_sample_ (13).jpg,100k,0.955743,0.005408,0.040059,0.101661,0.146579,0.229265,0.023192,0.0,...,0.991997,0.981498,0.971684,0.982448,0.981522,0.981498,0.955160,0.958117,0.971849,0.956316


In [107]:
feature_df.head()

,filename,label,B_Q1_B1,B_Q1_B2,B_Q1_B3,B_Q1_B4,B_Q1_B5,B_Q1_B6,B_Q1_B7,B_Q1_B8,...,correlation_d1_a90,correlation_d1_a135,correlation_d2_a0,correlation_d2_a45,correlation_d2_a90,correlation_d2_a135,correlation_d3_a0,correlation_d3_a45,correlation_d3_a90,correlation_d3_a135
0,100000_sample_ (1).jpg,100k,0.911242,0.017922,0.014839,0.014743,0.096645,0.155326,0.367985,0.0,...,0.991089,0.979421,0.967998,0.979754,0.980559,0.979421,0.950121,0.954202,0.971901,0.953769
1,100000_sample_ (10).jpg,100k,0.862024,0.010829,0.037429,0.105663,0.333602,0.364512,0.008516,0.0,...,0.990465,0.978796,0.968332,0.979177,0.981332,0.978796,0.951530,0.955657,0.973292,0.954916
2,100000_sample_ (11).jpg,100k,0.954914,0.004058,0.028219,0.116932,0.146349,0.225933,0.034489,0.0,...,0.991861,0.980943,0.971543,0.982211,0.981542,0.980943,0.955597,0.958445,0.972042,0.956086
3,100000_sample_ (12).jpg,100k,0.774196,0.004858,0.015992,0.024697,0.156378,0.608102,0.074090,0.0,...,0.991168,0.979711,0.968457,0.979057,0.982071,0.979711,0.951477,0.954519,0.973748,0.955960
4,100000_sample_ (13).jpg,100k,0.955743,0.005408,0.040059,0.101661,0.146579,0.229265,0.023192,0.0,...,0.991997,0.981498,0.971684,0.982448,0.981522,0.981498,0.955160,0.958117,0.971849,0.956316


In [108]:
feature_df.to_csv(f"features/{FEATURES_NAME}.csv", index=False)

print("Berhasil disimpan!")

Berhasil disimpan!
